# LCProp 0.4.0rc1 — hosted CPU qualification

Separate test notebook; the production 0.3.0 notebook is unchanged. Use a **fresh hosted CPU runtime**. No manual uploads or repository checkout are needed. Stage 3E independently verified the exact TestPyPI artifacts on 2026-10-17. Setup rechecks their identities before installation and stops on any mismatch.

Engine source: `2e195579dc3d76f6477bf6c885fd537043acdf9d`.
Wheel SHA-256: `2cc0ede68970b2d7de08dc385597861ab1706ad7b08fa4fa2169c5539f8e396e`.
Sdist SHA-256: `7f3367ea5a2940b8cd2256ec9c93ab44bfae2f3907798dda1158c96b1fdf2738` (recorded, not installed here).

Run setup, then initialize. Installation uses the hash-verified TestPyPI wheel as a local requirement and resolves its dependencies exclusively from production PyPI. Neither cell runs propagation. **Only the interface's Run button executes science.** Unfamiliar-notebook/custom-widget warnings are normal Colab trust prompts; review this notebook before running it.


In [ ]:
#@title 1. Install and verify the exact RC1 (details collapsed)
import os, sys, json, hashlib, tempfile, subprocess, zipfile, urllib.request, urllib.error
from pathlib import Path
from importlib import metadata

RC_VERSION = "0.4.0rc1"
RC_SOURCE = "2e195579dc3d76f6477bf6c885fd537043acdf9d"
WHEEL_NAME = "lcprop-0.4.0rc1-py3-none-any.whl"
WHEEL_SHA = "2cc0ede68970b2d7de08dc385597861ab1706ad7b08fa4fa2169c5539f8e396e"
SDIST_SHA = "7f3367ea5a2940b8cd2256ec9c93ab44bfae2f3907798dda1158c96b1fdf2738"
_rc_ready = False
loaded = any(n == "lcprop" or n.startswith("lcprop.") for n in sys.modules)
if loaded and (globals().get("_rc_verified") != (RC_VERSION, WHEEL_SHA) or metadata.version("lcprop") != RC_VERSION):
    raise RuntimeError("Another or unverified LCProp is loaded. Export results, restart the runtime, then run setup.")
print("Checking exact TestPyPI release…")
try:
    with urllib.request.urlopen("https://test.pypi.org/pypi/lcprop/0.4.0rc1/json", timeout=30) as response:
        release = json.load(response)
except urllib.error.HTTPError as error:
    raise RuntimeError("RC1 is not available from TestPyPI (HTTP %s). Stop: do not substitute another source." % error.code) from error
files = {f["filename"]: f for f in release["urls"]}
expected = {WHEEL_NAME: WHEEL_SHA, "lcprop-0.4.0rc1.tar.gz": SDIST_SHA}
if release["info"]["version"] != RC_VERSION or set(files) != set(expected):
    raise RuntimeError("Unexpected TestPyPI release inventory; stop for review.")
for name, digest in expected.items():
    if files[name]["digests"]["sha256"] != digest:
        raise RuntimeError("Published artifact hash differs: " + name)
with tempfile.TemporaryDirectory(prefix="lcprop-rc1-") as temporary:
    wheel = Path(temporary) / WHEEL_NAME
    with urllib.request.urlopen(files[WHEEL_NAME]["url"], timeout=60) as response:
        wheel.write_bytes(response.read())
    if hashlib.sha256(wheel.read_bytes()).hexdigest() != WHEEL_SHA:
        raise RuntimeError("Downloaded wheel hash mismatch; nothing installed.")
    if not loaded:
        before = {n: metadata.version(n) for n in ("numpy", "scipy", "matplotlib", "ipywidgets", "IPython", "ipympl") if n in sys.modules}
        env = {k:v for k,v in os.environ.items() if not k.startswith("PIP_")}
        env["PIP_CONFIG_FILE"] = os.devnull  # disable extra indexes from pip configuration
        print("Installing verified wheel; dependencies from production PyPI…")
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple",
                                 str(wheel) + "[colab]"], env=env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Installation failed. Retry in a fresh CPU runtime.\n" + result.stderr[-5000:])
        changed = [n for n,v in before.items() if metadata.version(n) != v]
        if changed:
            raise RuntimeError("Loaded dependency versions changed: " + ", ".join(changed) + ". Restart the runtime and rerun setup; no automatic restart.")
    if metadata.version("lcprop") != RC_VERSION:
        raise RuntimeError("Installed LCProp version mismatch.")
    distribution = metadata.distribution("lcprop")
    with zipfile.ZipFile(wheel) as archive:
        payload = [n for n in archive.namelist() if n.startswith("lcprop/") and not n.endswith("/")]
        if len(payload) != 269:
            raise RuntimeError("Unexpected runtime-file inventory.")
        for name in payload:
            if Path(distribution.locate_file(name)).read_bytes() != archive.read(name):
                raise RuntimeError("Installed runtime identity mismatch: " + name)
import lcprop
if Path(lcprop.__file__).resolve() != Path(distribution.locate_file("lcprop/__init__.py")).resolve():
    raise RuntimeError("LCProp import is shadowed by another source; restart without that checkout on sys.path.")
from lcprop.runners.distribution_deployment import resolve_installed_distribution
resolve_installed_distribution(lcprop.__file__)
check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
if check.returncode:
    raise RuntimeError("Dependency consistency failed; record this before proceeding.\n" + check.stdout + check.stderr)
_rc_verified = (RC_VERSION, WHEEL_SHA)
_rc_ready = True
print("Ready: LCProp 0.4.0rc1; 269 installed files verified. No scientific execution.")


In [ ]:
#@title 2. Open the interface
MODE = "interactive"  # Change to "static" in a fresh runtime for the fallback.
if not globals().get("_rc_ready", False):
    raise RuntimeError("Complete verified setup first.")
from IPython.display import clear_output, display
from lcprop.notebook import InteractiveLC
if MODE not in ("interactive", "static"):
    raise ValueError("Choose interactive or static.")
if globals().get("app_rc1") is not None:
    if not isinstance(app_rc1, InteractiveLC) or globals().get("_rc_mode") != MODE or app_rc1.disposed:
        raise RuntimeError("Export needed results and restart before changing mode or replacing this controller.")
else:
    if MODE == "interactive":
        from google.colab import output
        output.enable_custom_widget_manager()
        get_ipython().run_line_magic("matplotlib", "widget")
    else:
        get_ipython().run_line_magic("matplotlib", "inline")
    app_rc1 = InteractiveLC()
    _rc_mode = MODE
clear_output(wait=True)
display(app_rc1.widget)
print("Ready — " + MODE + ". Only Run executes the solver; repeated initialization preserves this controller.")


## Hosted acceptance checklist

Record PASS/FAIL and screenshots for each; backend transmission alone is not browser-paint evidence.

- Initial canvas is complete. First Beam 2 activation, then off/on repeatedly, shows both spots, markers, arrows, axes and colorbar without an extra plot click or manual refresh.
- Select either beam, including overlapping centers. Move/Tilt affects only the selected geometry; numeric coordinates/angles agree after release. Try a radius edit and coherent/incoherent selection. Editing never starts propagation.
- Canvas remains stable without duplicate views, clipped colorbars or intermittent white regions. Cancel a gesture and confirm recovery.
- Rerun initialization: same controller, controls and completed record (if any), one displayed interface. Do not switch MODE underneath it.
- If widgets are blank or unstable: record failure, export any needed records, restart, set MODE to `static`, and rerun setup/initialization. Static fallback must show the same scientific launch content; mouse editing is unavailable. Static success does not qualify the interactive backend.
- **Completed-run isolation is pending unless separately authorized:** after an authorized bounded Run, record request identity; edit the draft and verify executed parameters, completed plots and exports remain attached to that run. A subsequent Run must capture the new draft. Do not run a calculation merely to test graphics. Do not claim this check passed without a completed run.

This notebook does not execute any solver automatically. Scientific execution and cross-platform array comparisons require a separate explicit qualification decision. There is no GPU or Slurm setup. Runtime restart loses unexported in-memory results.


In [ ]:
#@title 3. Record environment and setup provenance (no solver)
import platform
if not globals().get("_rc_ready", False):
    raise RuntimeError("Verified setup has not completed.")
qualification_environment = {
    "expected_release_source": RC_SOURCE, "version": metadata.version("lcprop"),
    "wheel_sha256": WHEEL_SHA, "sdist_sha256": SDIST_SHA,
    "installed_runtime_files_verified": 269, "module": lcprop.__file__,
    "python": sys.version, "platform": platform.platform(), "mode": globals().get("_rc_mode"),
    "dependencies": {n: metadata.version(n) for n in ("numpy", "scipy", "matplotlib", "ipywidgets", "IPython", "ipympl")},
    "browser_observations": "Record manually; not inferred from setup success.",
}
print(json.dumps(qualification_environment, indent=2))
